In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm
import time, random, os, sys
from typing import Callable, Dict, List, Set, Any, Optional, Union

# Dynamic path allocation
current_dir = os.getcwd()
project_path = os.path.abspath(os.path.join(current_dir, "..", ".."))
sys.path.append(project_path)

from algorithms.compute_PC import compute_pc

# 1. Affinity Score Results

In [2]:
greedy_affine = "/home/tuguldur/Development/Research/Dev/ECNDP/ECNDP/python/results/csv/Result_ECNDP_ER_all_aff_only.csv"
others = "/home/tuguldur/Development/Research/Dev/ECNDP/ECNDP/python/results/csv/Result_ECNDP_random_ER_all_affinity.csv"


In [4]:
df_1 = pd.read_csv(greedy_affine)
df_2 = pd.read_csv(others)
df_1

,case,algo,terminals,non_terminals,K,obj,time
0,2,Greedy affinity - no ls,9,91,9,36.0,0.02034
1,2,Greedy affinity - ls,9,91,9,21.0,0.32348
2,2,Greedy affinity - no ls,9,91,18,36.0,0.02088
3,2,Greedy affinity - ls,9,91,18,21.0,0.56133
4,2,Greedy affinity - no ls,9,91,27,36.0,0.02039
5,2,Greedy affinity - ls,9,91,27,21.0,0.72303
6,2,Greedy affinity - no ls,9,91,36,36.0,0.02226
7,2,Greedy affinity - ls,9,91,36,21.0,0.81405
8,2,Greedy affinity - no ls,9,91,45,36.0,0.02094
9,2,Greedy affinity - ls,9,91,45,21.0,0.83899


In [5]:
df_affinity = df_1[df_1["algo"].str.contains("Greedy affinity")]

# from df_2 keep only ES and MIS
df_other = df_2[
  df_2["algo"].str.contains("Greedy ES") |
  df_2["algo"].str.contains("Greedy MIS")
]


In [6]:
# combine
df_all = pd.concat([df_affinity, df_other], ignore_index=True)

# optional: sort
df_all = df_all.sort_values(["terminals", "algo", "K"])

In [8]:
df_all["K"] = pd.to_numeric(df_all["K"], errors="coerce")

# keep only integer K values
df_all = df_all[df_all["K"] % 1 == 0]

# convert to int for clean plotting
df_all["K"] = df_all["K"].astype(int)

In [12]:
def clean_name(algo_name):
  if "ES" in algo_name:
    base = "Greedy"
  elif "MIS" in algo_name:
    base = "MIS"
  else:
    base = "Affinity"

  if "no ls" in algo_name:
    return base + " (no LS)"
  else:
    return base + " (LS)"

In [14]:
import os
import matplotlib.pyplot as plt

# create folder for plots
output_folder = "/home/tuguldur/Development/Research/Dev/ECNDP/ECNDP/python/results/plots"
os.makedirs(output_folder, exist_ok=True)

terminal_values = [9, 19, 30, 40, 50]

for terminal in terminal_values:
  df_terminal = df_all[df_all["terminals"] == terminal]

  plt.figure()

  for algo_name in df_terminal["algo"].unique():
    df_algo = df_terminal[df_terminal["algo"] == algo_name]
    df_algo = df_algo.sort_values("K")

    plt.plot(
      df_algo["K"],
      df_algo["obj"],
      marker="o",
      label=clean_name(algo_name)
    )

  k_values = sorted(df_terminal["K"].unique())
  plt.xticks(k_values)

  plt.title(f"Objective vs K (terminals = {terminal})")
  plt.xlabel("K")
  plt.ylabel("Objective (obj)")
  plt.legend()
  plt.grid()

  # save file
  filename = f"{output_folder}/terminals_{terminal}.png"
  plt.savefig(filename, dpi=300, bbox_inches="tight")

  plt.close()  # important to avoid overlap

# 2. Idea visualizations

## 2.1 Graph reducing on neighborhood similarity

### 1. Greedy algorithm - only for case 2

In [ ]:
def graph_reduce(
    G: nx.Graph, s_neigh: Set, t_neigh: Set, s: int, t: int
    ):
  
  # Get rest of the grapg G_rest = G[V \ {Neig(s) U Neig{t} U s U t}]
  s_t_and_neighbors = s_neigh | t_neigh | s | t

  G.remove_nodes_from(s_t_and_neighbors)

  


In [ ]:
def neighborhood_metric(
    G: nx.Graph, terminals: Set, 
    dist_diff: int, normalize_fn: Callable):

  T : set = set(terminals)
  V : set = set(G.nodes())

  # Iterate over dist_diff of values 0, 1, 2, and so on
  # Neighborhood_similarity == 0, 1, 2 and so on.
  
  # Combination between any two terminals
  for s in T:
    for t in T - {s}:

      # if s == t: 
      #   continue

      # Ensure there are no terminals in the middle

      # There will be unique neighbors
      s_neighbors_no_ters : Set = set()
      t_neighbors_no_ters : Set = set()

      s_neighbors = set(G.neighbors(s))
      t_neighbors = set(G.neighbors(t))

      for s_nei in s_neighbors:
        if s_nei["terminal"] == False:
          s_neighbors_no_ters.add(s_nei)

      for t_nei in t_neighbors:
        if t_nei["terminal"] == False:
          t_neighbors_no_ters.add(t_nei)

      # Metric: (|A U B| - |A disjoint B|) / |A U B|
      edit_dist = (s_neighbors_no_ters | t_neighbors_no_ters) - \
        (s_neighbors_no_ters - t_neighbors_no_ters)

      metric_diff = normalize_fn(edit_dist)

      if metric_diff == dist_diff:
        # Reduce the subgraph and update
        G_reduced = graph_reduce(
          G.copy(), s_neighbors_no_ters, t_neighbors_no_ters, s, t)
  
  return G_reduced

In [ ]:
# 1. Get find neighbors of all terminal nodes
# 2. Get terminal-terminal blocks (connected components) from the terminals induces subgraph
# which is H_t = G[T]
# 3. iterate over itertools.combinations(terminals, 2)
# but only between any two terminals that have not in the same blocks
# 4. Find the neighbors difference metric by |A U B| - |A disjoint|,
# where A = A_s \ T and B = B_s \ T, A_s and B_s are neighbors of 
# any two terminals from the different block
# 5. Normalize the metric
# 6. Reduce the graph

def find_nonterminal_neighbors(
  G: nx.Graph, t: int
) -> set:
  return {n for n in G.neighbors(t) if n["terminal"] == False}

def neighborhood_metric(
  G: nx.Graph, t: int, s: int, 
  terminals: set[int], normalize_fn: Optional[Callable[[set[int], set[int], int], float]] = None
) -> int | float:
  
  # A = Neigh(t) \ T, B = Neigh(s) \ T
  A = find_nonterminal_neighbors(G, t)
  B = find_nonterminal_neighbors(G, s)

  # |A U B| size
  # Symmetric difference size
  raw = len(A ^ B)

  if normalize_fn is None:
    return raw
  
  union_size = len(A | B)
  return normalize_fn(A, B, union_size)

from collections import defaultdict

def get_terminal_blocks(
    G: nx.Graph, terminals: set[int],
) -> dict[int, int]:
  
  # 1. Get terminals induced subgraph H_t = G[T]
  H_t = G.subgraph(terminals)

  # 2. Find connected components of the subgraph
  terminal_blocks = nx.connected_components(H_t)

  # 3. Create a map -> dictionary
  term_blocks = defaultdict(int)

  for i, block in enumerate(terminal_blocks):
    for t in block:
      term_blocks[t] = i

  return term_blocks
  
def reduce_graph(
    G: nx.Graph, terminals: set[int],
)
